# Preparation

In [2]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
print(physical_devices[0])

PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [ ]:
# === Phase 1: Global Config and Imports ===
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
import zipfile

# Configuration dictionary
CONFIG = {
    'SEED': 42,
    'INPUT_SIZE': 224,
    'BATCH_SIZE': 32,
    'TEST_SIZE': 0.2,
    'KFOLD': 3,
    'ZIP_PATH': 'train_images.zip',
    'IMAGE_DIR': 'unzipped/train_images',
    'CSV_PATH': 'meta_train.csv',
    'TRAIN_CSV': 'train.csv',
    'TEST_CSV': 'test.csv',
    'MODEL_DIR': 'saved_models/',
    'HISTORY_DIR': 'saved_histories/',
    'EPOCHS': 50,
    'EARLY_STOP_PATIENCE': 6,
    'LEARNING_RATE': 1e-3
}

# === Phase 1.5: Unzip Train Images ===
def unzip_train_images():
    if not os.path.exists('unzipped'):
        os.makedirs('unzipped')

    with zipfile.ZipFile(CONFIG['ZIP_PATH'], 'r') as zip_ref:
        zip_ref.extractall('unzipped')

    print(f"Unzipped {CONFIG['ZIP_PATH']} successfully.")

unzip_train_images()

# Set random seeds
random.seed(CONFIG['SEED'])
np.random.seed(CONFIG['SEED'])
tf.random.set_seed(CONFIG['SEED'])

# === Phase 2: Load Metadata ===
meta = pd.read_csv(CONFIG['CSV_PATH'])

# === Phase 3: Validate Images (missing or corrupt) ===
def validate_images(meta):
    missing = []
    corrupt = []
    for iid, label in zip(meta['image_id'], meta['label']):
        fname = iid if iid.lower().endswith('.jpg') else f"{iid}.jpg"
        fpath = os.path.join(CONFIG['IMAGE_DIR'], label, fname)
        if not os.path.isfile(fpath):
            missing.append(fpath)
        else:
            try:
                with Image.open(fpath) as img:
                    img.verify()
            except:
                corrupt.append(fpath)
    return missing, corrupt

missing_files, corrupt_files = validate_images(meta)
print(f"Missing files: {len(missing_files)}")
print(f"Corrupted files: {len(corrupt_files)}")

# === Phase 4: Summarize Image Resolutions BEFORE Rotation ===
def summarize_resolution(meta, title="Resolution Summary"):
    resolutions = []
    for iid, label in zip(meta['image_id'], meta['label']):
        fname = iid if iid.lower().endswith('.jpg') else f"{iid}.jpg"
        fpath = os.path.join(CONFIG['IMAGE_DIR'], label, fname)
        with Image.open(fpath) as img:
            w, h = img.size
            resolutions.append((w, h))
    df_res = pd.DataFrame(resolutions, columns=['width', 'height'])
    summary = df_res.groupby(['width', 'height']).size().reset_index(name='count')
    print(f"\n=== {title} ===")
    print(summary)
    return summary

summarize_resolution(meta, title="Resolution BEFORE Rotation")

# === Phase 5: Rotate Landscape Images ===
def rotate_landscape(meta):
    landscapes = []
    for iid, label in zip(meta['image_id'], meta['label']):
        fname = iid if iid.lower().endswith('.jpg') else f"{iid}.jpg"
        fpath = os.path.join(CONFIG['IMAGE_DIR'], label, fname)
        with Image.open(fpath) as img:
            w, h = img.size
            if w > h:
                landscapes.append(fpath)

    for path in landscapes:
        img = Image.open(path)
        img_rotated = img.transpose(Image.ROTATE_90)
        img_rotated.save(path)

    print(f"Rotated {len(landscapes)} landscape images.")

rotate_landscape(meta)

# === Phase 6: Summarize Image Resolutions AFTER Rotation ===
summarize_resolution(meta, title="Resolution AFTER Rotation")

# === Phase 7: Add Age Bins for Stratified Split ===
meta['age_bin'] = pd.qcut(meta['age'], q=5, labels=False, duplicates='drop')

# === Phase 8: Train/Test Split with Stratification ===
train_val, test = train_test_split(
    meta,
    test_size=CONFIG['TEST_SIZE'],
    stratify=meta['age_bin'],
    random_state=CONFIG['SEED']
)

# === Phase 9: Scale Age ===
age_mean = train_val['age'].mean()
age_std = train_val['age'].std()

train_val['age_scaled'] = (train_val['age'] - age_mean) / age_std
test['age_scaled'] = (test['age'] - age_mean) / age_std

# === Phase 10: Save Clean CSVs for tf.data Loading ===
def safe_image_path(row):
    fname = row['image_id'] if row['image_id'].lower().endswith('.jpg') else f"{row['image_id']}.jpg"
    return os.path.join(CONFIG['IMAGE_DIR'], row['label'], fname)

train_val['image_path'] = train_val.apply(safe_image_path, axis=1)
test['image_path'] = test.apply(safe_image_path, axis=1)

train_val[['image_path', 'age_scaled']].to_csv(CONFIG['TRAIN_CSV'], index=False)
test[['image_path', 'age_scaled']].to_csv(CONFIG['TEST_CSV'], index=False)

# === Phase 11: Compute RGB Mean and Std for Normalization ===
def compute_rgb_stats(image_paths):
    sum_rgb = np.zeros(3)
    sum_sq = np.zeros(3)
    total_pixels = 0

    for path in image_paths:
        img = np.array(Image.open(path).convert('RGB'), dtype=np.float32) / 255.0
        sum_rgb += img.sum(axis=(0, 1))
        sum_sq += (img ** 2).sum(axis=(0, 1))
        total_pixels += img.shape[0] * img.shape[1]

    mean_rgb = sum_rgb / total_pixels
    std_rgb = np.sqrt(sum_sq / total_pixels - mean_rgb**2)
    return mean_rgb, std_rgb

MEAN, STD = compute_rgb_stats(train_val['image_path'])
print(f"Image RGB Mean: {MEAN}, Std: {STD}")

# === Phase 12: Define tf.data Pipelines ===
def parse_csv_line(line):
    parts = tf.strings.split(line, sep=",")
    path = parts[0]
    label = tf.strings.to_number(parts[1], tf.float32)
    return path, label

def preprocess_image(path, label, augment=False):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)

    h, w = tf.shape(img)[0], tf.shape(img)[1]
    frac = tf.cast(tf.minimum(h, w), tf.float32) / tf.cast(tf.maximum(h, w), tf.float32)
    img = tf.image.central_crop(img, central_fraction=frac)

    img = tf.image.resize(img, [CONFIG['INPUT_SIZE'], CONFIG['INPUT_SIZE']])

    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.1)

        zoom_factor = tf.random.uniform([], 0.9, 1.1)
        crop_size = tf.cast(zoom_factor * tf.cast(tf.shape(img)[:2], tf.float32), tf.int32)
        img = tf.image.resize_with_crop_or_pad(img, crop_size[0], crop_size[1])
        img = tf.image.resize(img, [CONFIG['INPUT_SIZE'], CONFIG['INPUT_SIZE']])

        k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
        img = tf.image.rot90(img, k)

    img = (img - MEAN) / STD
    return img, label

def make_dataset(csv_file, augment=False, shuffle=False):
    ds = tf.data.TextLineDataset([csv_file]) \
        .skip(1) \
        .map(parse_csv_line, num_parallel_calls=tf.data.AUTOTUNE) \
        .map(lambda path, age: preprocess_image(path, age, augment), num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1024, seed=CONFIG['SEED'])
    return ds.batch(CONFIG['BATCH_SIZE']).prefetch(tf.data.AUTOTUNE)

Unzipped train_images.zip successfully.
Missing files: 0
Corrupted files: 0

=== Resolution BEFORE Rotation ===
   width  height  count
0    480     640  10403
1    640     480      4
Rotated 4 landscape images.

=== Resolution AFTER Rotation ===
   width  height  count
0    480     640  10407
Image RGB Mean: [0.49692728 0.58833559 0.22903944], Std: [0.24089137 0.24288366 0.2172656 ]


# Modeling

In [ ]:
# === Imports ===
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50, MobileNetV2, EfficientNetB0, EfficientNetB1, DenseNet121, InceptionV3

# === Model Builder ===
def build_model(model_name, input_shape=(224, 224, 3)):
    """Build model architecture based on the given name."""
    if model_name == "ResNet50":
        base = ResNet50(weights=None, include_top=False, input_shape=input_shape)
    elif model_name == "MobileNetV2":
        base = MobileNetV2(weights=None, include_top=False, input_shape=input_shape)
    elif model_name == "EfficientNetB0":
        base = EfficientNetB0(weights=None, include_top=False, input_shape=input_shape)
    elif model_name == "EfficientNetB1":
        base = EfficientNetB1(weights=None, include_top=False, input_shape=input_shape)
    elif model_name == "DenseNet121":
        base = DenseNet121(weights=None, include_top=False, input_shape=input_shape)
    elif model_name == "InceptionV3":
        base = InceptionV3(weights=None, include_top=False, input_shape=input_shape)
    elif model_name == "CustomCNN":
        return build_custom_cnn(input_shape)
    else:
        raise ValueError(f"Unknown model name: {model_name}")

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(1)(x)

    model = models.Model(inputs=base.input, outputs=output)

    model.compile(
        optimizer=optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE']),
        loss='mse',
        metrics=[tf.keras.metrics.MeanAbsoluteError()]
    )
    return model

# === Custom CNN Builder ===
def build_custom_cnn(input_shape=(224, 224, 3)):
    """Build custom lightweight CNN."""
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1)
    ])
    
    model.compile(
        optimizer=optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE']),
        loss='mse',
        metrics=[tf.keras.metrics.MeanAbsoluteError()]
    )
    return model

# === Model Trainer ===
def train_and_save_model(model, model_name, train_ds, val_ds):
    """Train the model and save weights and training history."""
    os.makedirs(CONFIG['MODEL_DIR'], exist_ok=True)
    os.makedirs(os.path.join(CONFIG['HISTORY_DIR'], model_name), exist_ok=True)

    early_stop = callbacks.EarlyStopping(monitor='val_mean_absolute_error', patience=CONFIG['EARLY_STOP_PATIENCE'], restore_best_weights=True)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CONFIG['EPOCHS'],
        callbacks=[early_stop],
        verbose=1
    )

    # Save model and history
    model.save(os.path.join(CONFIG['MODEL_DIR'], f"{model_name}.h5"))
    pd.DataFrame(history.history).to_csv(os.path.join(CONFIG['HISTORY_DIR'], model_name, "history.csv"), index=False)

    return history

# === Model Evaluator ===
def evaluate_model(model, test_ds, age_mean, age_std):
    """Evaluate the model and return metrics and predictions."""
    test_df = pd.read_csv(CONFIG['TEST_CSV'])
    true_scaled = test_df["age_scaled"].values
    true_unscaled = true_scaled * age_std + age_mean

    y_pred_scaled = model.predict(test_ds).flatten()
    y_pred_unscaled = y_pred_scaled * age_std + age_mean

    mae = mean_absolute_error(true_unscaled, y_pred_unscaled)
    mse = mean_squared_error(true_unscaled, y_pred_unscaled)
    r2 = r2_score(true_unscaled, y_pred_unscaled)

    return mae, mse, r2, true_unscaled, y_pred_unscaled

# === Training History Plotter ===
def plot_training_history(model_name):
    """Plot training loss and MAE curves."""
    history_csv_path = os.path.join(CONFIG['HISTORY_DIR'], model_name, "history.csv")
    history = pd.read_csv(history_csv_path)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))

    ax1.plot(history['loss'], label='train loss')
    ax1.plot(history['val_loss'], label='val loss')
    ax1.set_title(f'{model_name} - Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('MSE Loss')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(history['mean_absolute_error'], label='train MAE')
    ax2.plot(history['val_mean_absolute_error'], label='val MAE')
    ax2.set_title(f'{model_name} - Mean Absolute Error')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('MAE')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['HISTORY_DIR'], model_name, "training_plot.png"))
    plt.close()

# === Predicted vs Actual Plot ===
def plot_predicted_vs_actual(true_age, predicted_age, model_name):
    """Plot predicted vs true age scatter plot."""
    plt.figure(figsize=(6,6))
    plt.scatter(true_age, predicted_age, alpha=0.5)
    min_val = min(true_age.min(), predicted_age.min())
    max_val = max(true_age.max(), predicted_age.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')
    plt.xlabel("True Age (days)")
    plt.ylabel("Predicted Age (days)")
    plt.title(f"{model_name} - Predicted vs Actual")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['HISTORY_DIR'], model_name, "predicted_vs_actual.png"))
    plt.close()

# === Residual Plot ===
def plot_residuals(true_age, predicted_age, model_name):
    """Plot residuals (true - predicted) vs true age."""
    residuals = true_age - predicted_age

    plt.figure(figsize=(6,5))
    plt.scatter(true_age, residuals, alpha=0.5)
    plt.axhline(0, color='red', linestyle='--')
    plt.xlabel("True Age (days)")
    plt.ylabel("Residual (True - Predicted)")
    plt.title(f"{model_name} - Residual Plot")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['HISTORY_DIR'], model_name, "residuals.png"))
    plt.close()

# === Main Training Loop ===
def main_training_loop():
    """Train, evaluate, and save results for all baseline models."""
    train_ds = make_dataset(CONFIG['TRAIN_CSV'], augment=True, shuffle=True)
    val_ds = make_dataset(CONFIG['TEST_CSV'], augment=False)
    test_ds = make_dataset(CONFIG['TEST_CSV'], augment=False)

    model_names = [
        "ResNet50", "MobileNetV2", "EfficientNetB0", "EfficientNetB1",
        "DenseNet121", "InceptionV3", "CustomCNN"
    ]

    results = []

    for model_name in model_names:
        print(f"\n🚀 Training {model_name}...")

        model = build_model(model_name)
        history = train_and_save_model(model, model_name, train_ds, val_ds)

        mae, mse, r2, true_unscaled, y_pred_unscaled = evaluate_model(model, test_ds, age_mean, age_std)
        print(f"✅ {model_name} - MAE: {mae:.2f}, MSE: {mse:.2f}, R²: {r2:.3f}")

        plot_training_history(model_name)
        plot_predicted_vs_actual(true_unscaled, y_pred_unscaled, model_name)
        plot_residuals(true_unscaled, y_pred_unscaled, model_name)

        results.append({
            'Model': model_name,
            'Validation MAE': mae,
            'Validation MSE': mse,
            'Validation R2': r2
        })

    pd.DataFrame(results).to_csv(os.path.join(CONFIG['HISTORY_DIR'], "baseline_model_comparison.csv"), index=False)
    print("\n🏆 Baseline model comparison completed. Check saved results.")

# Execute the training code

In [12]:
main_training_loop()


🚀 Training ResNet50...
Epoch 1/50
261/261 [==============================] - 145s 512ms/step - loss: 2.1417 - mean_absolute_error: 0.9418 - val_loss: 0.9985 - val_mean_absolute_error: 0.8581
Epoch 2/50
261/261 [==============================] - 134s 501ms/step - loss: 0.9226 - mean_absolute_error: 0.8007 - val_loss: 0.8416 - val_mean_absolute_error: 0.7732
Epoch 3/50
 59/261 [=====>........................] - ETA: 1:33 - loss: 0.9327 - mean_absolute_error: 0.8039

: 